# 04 · Módulo de degradación espectral (super-resolución)

**Qué hace:** calibra un módulo de degradación específico de dominio (MTF Butterworth + grano correlacionado + escaneo JPEG) minimizando la distancia log-RAPSD frente al espectro del deterioro histórico real; exporta los rangos calibrados y valida (ajuste espectral, regresión cromática, inspección visual).

**Qué necesita:**
- `artifacts/psd_objetivo.npz` (objetivo espectral medido en la Fase 4a — viene en el repo)
- Kaggle: `joe1995/div2k-dataset` (parches HR para calibrar)
- Release `v1.0` `vintage_degraded.zip` (fotografías históricas para las figuras de inspección visual)

**Qué deja escrito:** `ROOT/_out/04/parametros_degradacion.json` y figuras en `ROOT/_out/04/`. El módulo `dataset_espectral.py` versionado en el repo es la fuente de verdad; recalibrar solo regenera el JSON.

**Arranque:** primera celda de código.

---

## Contexto de la Fase 4a

La Fase 4a estableció que la degradación de la tercera entrega no reproduce el espectro del
deterioro real, con una distancia log-RAPSD de 0,911 frente a un umbral de 0,10. El signo
del hueco resultó además contrario al previsto: **la simulación conserva más potencia que
el material histórico en todo el rango**, y en la banda alta lo hace por 1,38 unidades
logarítmicas con separación intercuartílica total. La lectura es que el ruido y la
compresión del pipeline anterior inyectan energía de banda ancha que compensa el efecto del
desenfoque, de modo que el resultado neto no es una atenuación sino un suelo de ruido.

Esta fase construye el módulo que corrige ese comportamiento. La cadena respeta el orden
físico del proceso fotográfico y sus parámetros se calibran minimizando la distancia
espectral frente al objetivo medido en la Fase 4a.

| Etapa | Componente | Efecto espectral buscado |
|---|---|---|
| Óptica | MTF Butterworth, con anisotropía opcional | Atenuación controlada de alta frecuencia |
| Emulsión | Grano correlacionado acromático | Textura con caída propia, no ruido blanco |
| Reproducción | Trama de semitono o bandeado | Impulsos discretos, solo en parte del material |
| Digitalización | Submuestreo, ruido de sensor y compresión | Suelo de ruido contenido |

## 1. Configuración

In [ ]:
!pip -q install scipy
from colab_setup import aplicar_parches, preparar_repo, descargar_datos
aplicar_parches()
ROOT = preparar_repo()
DATA = descargar_datos("kaggle:div2k", "release:vintage_degraded", root=ROOT)

In [ ]:
import os, sys, json, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import qmc

PSD_OBJETIVO = ROOT / "artifacts" / "psd_objetivo.npz"
DIV2K = DATA["kaggle:div2k"]

SALIDA = ROOT / "_out" / "04"
SALIDA.mkdir(parents=True, exist_ok=True)
RESULTADOS = MODULO_DIR = SALIDA          # alias: todo se escribe en ROOT/_out/04

ESCALA, ANALISIS, SEMILLA = 4, 256, 1234
LADO_HR = ANALISIS*ESCALA          # 1024: recorte HR que produce un parche LQ de 256
N_CALIB = 12                       # parches usados en cada evaluacion de la calibracion
N_SONDEO, N_REFINO = 32, 70        # presupuesto de la busqueda gruesa y del refinamiento

COLORES = {'objetivo':'#000000', 'inicial':'#0072B2', 'calibrado':'#E69F00'}

plt.rcParams.update({'font.family':'sans-serif','font.size':8,'axes.labelsize':9,
                     'xtick.labelsize':7,'ytick.labelsize':7,'figure.dpi':120,'savefig.dpi':300})

EXT = ('.png','.jpg','.jpeg','.tif','.tiff','.bmp')
def listar(c, ext=EXT):
    c = Path(c); return [r for r in sorted(c.rglob('*')) if r.suffix.lower() in ext] if c.exists() else []

assert PSD_OBJETIVO.exists(), f'Falta el objetivo de la Fase 4a en {PSD_OBJETIVO}'
print('Salida  :', SALIDA)
print('Objetivo:', PSD_OBJETIVO.name)
print('DIV2K   :', len(listar(DIV2K)), 'imágenes')

### Qué banda se usa como objetivo

La Fase 4a midió el hueco en tres bandas y encontró solape intercuartílico del 100 % en la
banda baja, del 84 % en la media y del 0 % en la alta. Solo esta última presenta separación
inequívoca, y el informe de la Fase 4a ya advirtió de que la banda baja no debía usarse
todavía como criterio de calibración: con veintiocho fotografías de origen, la mediana
histórica en esa banda tiene demasiada varianza.

Hay además una razón estructural. La RAPSD se normaliza por el contraste de cada parche, de
modo que la potencia total queda fijada; forzar la caída de alta frecuencia desplaza
necesariamente el resto de la curva, y el módulo no dispone de ningún grado de libertad que
suba la potencia de baja frecuencia de forma independiente. Incluir la banda baja en el
objetivo obligaría al optimizador a repartir un error que no puede eliminar, degradando el
ajuste donde sí importa.

**La calibración se restringe por tanto a 0,10–0,45 ciclos por píxel.** La distancia sobre
la banda completa se sigue calculando para poder compararla con la Fase 4a, pero como
magnitud de informe y no como función objetivo. Se añade también una distancia de forma, que
descuenta el desplazamiento vertical medio y permite distinguir un error de forma de la
curva de un simple cambio de nivel.

In [ ]:
# --- Objetivo espectral medido en la Fase 4a ---------------------------------
Z = np.load(PSD_OBJETIVO)
FREQ = Z['freq']
OBJETIVO = Z['med_historicas']
PARTIDA  = Z['med_div2k_degradado']       # pipeline de la tercera entrega
LIMPIO   = Z['med_div2k_limpio']

BANDA_CAL     = (FREQ >= 0.10) & (FREQ < 0.45)    # función objetivo
BANDA_INFORME = (FREQ >= 0.02) & (FREQ < 0.45)    # comparable con la Fase 4a

def d_cal(perfil):
    return float(np.abs(perfil[BANDA_CAL] - OBJETIVO[BANDA_CAL]).mean())

def d_informe(perfil):
    return float(np.abs(perfil[BANDA_INFORME] - OBJETIVO[BANDA_INFORME]).mean())

def d_forma(perfil):
    '''Distancia tras descontar el desplazamiento vertical medio: aísla el error
    de forma de la curva de un simple cambio de nivel.'''
    r = perfil[BANDA_INFORME] - OBJETIVO[BANDA_INFORME]
    return float(np.abs(r - r.mean()).mean())

D_PARTIDA_CAL     = d_cal(PARTIDA)
D_PARTIDA_INFORME = d_informe(PARTIDA)
D_PARTIDA_FORMA   = d_forma(PARTIDA)

print(f'bins: {len(FREQ)}   banda de calibración: {BANDA_CAL.sum()} bins')
print(f'pipeline anterior · distancia de calibración (0,10–0,45): {D_PARTIDA_CAL:.4f}')
print(f'pipeline anterior · distancia de informe   (0,02–0,45): {D_PARTIDA_INFORME:.4f}')
print(f'pipeline anterior · distancia de forma                 : {D_PARTIDA_FORMA:.4f}')

## 2. Componentes del módulo

Las cuatro funciones siguientes son la implementación de referencia; el módulo exportado en
la sección 6 las reproduce. Dos decisiones merecen justificación.

**El MTF se aplica en Fourier y no por convolución.** Un Butterworth de orden ajustable no
tiene equivalente separable en el dominio espacial, y aplicarlo como núcleo 2D grande
dispararía el coste, tal como se comprobó en la Fase 4a. Multiplicar por la respuesta en
frecuencia es además la forma directa de controlar dónde empieza la caída y cuán abrupta es,
que son justamente los dos grados de libertad que la calibración necesita.

**El grano es acromático por construcción.** Un único campo de ruido se suma a los tres
canales, como corresponde a la película monocroma. Además de ser fiel al proceso
fotográfico, esto impide que el grano introduzca croma y protege frente a reeditar el
problema que las fases 0 a 3 dedicaron a diagnosticar. La modulación por `4·L·(1−L)` anula
el grano en negros y blancos saturados y lo hace máximo en tonos medios, que es el
comportamiento de la emulsión real.

In [ ]:
_MALLA = {}
def malla_rfft(h, w):
    '''Malla de frecuencias para rfft2, cacheada por tamaño.'''
    clave = (h, w)
    if clave not in _MALLA:
        fy = np.fft.fftfreq(h).astype(np.float32)[:, None]
        fx = np.fft.rfftfreq(w).astype(np.float32)[None, :]
        _MALLA[clave] = (fy, fx, np.sqrt(fy**2 + fx**2))
    return _MALLA[clave]


def H_butterworth(h, w, fc, orden, aniso=1.0, theta=0.0):
    '''Respuesta en frecuencia del paso bajo. fc en ciclos/píxel.'''
    fy, fx, _ = malla_rfft(h, w)
    ct, st = np.cos(theta), np.sin(theta)
    u, v = ct*fx + st*fy, -st*fx + ct*fy
    rho = np.sqrt((u*aniso)**2 + (v/aniso)**2)
    return (1.0/np.sqrt(1.0 + (rho/max(fc, 1e-4))**(2*orden))).astype(np.float32)


def mtf_butterworth(img, fc, orden, aniso=1.0, theta=0.0):
    h, w = img.shape[:2]
    H = H_butterworth(h, w, fc, orden, aniso, theta)
    if img.ndim == 2:
        return np.fft.irfft2(np.fft.rfft2(img)*H, s=(h, w)).astype(np.float32)
    salida = np.empty_like(img)
    for c in range(img.shape[2]):
        salida[..., c] = np.fft.irfft2(np.fft.rfft2(img[..., c])*H, s=(h, w))
    return salida


def grano_correlacionado(h, w, alpha, rng):
    '''Ruido con densidad espectral |f|^-alpha y varianza unidad.'''
    _, _, rho = malla_rfft(h, w)
    rho = rho.copy(); rho[0, 0] = 1.0
    F = np.fft.rfft2(rng.normal(size=(h, w)).astype(np.float32)) * (rho**(-alpha/2.0))
    g = np.fft.irfft2(F, s=(h, w)).astype(np.float32)
    g -= g.mean(); s = g.std()
    return g/s if s > 1e-8 else g


def aplicar_grano(img, sigma, alpha, rng):
    '''Grano ACROMÁTICO dependiente de la señal: el mismo campo en los tres canales.'''
    h, w = img.shape[:2]
    g = grano_correlacionado(h, w, alpha, rng)
    L = np.clip(img.mean(axis=2) if img.ndim == 3 else img, 0, 1)
    ruido = (sigma * 4.0*L*(1.0-L) * g).astype(np.float32)
    return img + (ruido[..., None] if img.ndim == 3 else ruido)


def artefacto_periodico(img, amplitud, freq, theta, fase=0.0):
    '''Trama de semitono o bandeado: impulsos discretos en el espectro.'''
    h, w = img.shape[:2]
    yy, xx = np.indices((h, w), dtype=np.float32)
    onda = amplitud*np.sin(2*np.pi*freq*(np.cos(theta)*xx + np.sin(theta)*yy) + fase)
    return img + (onda[..., None] if img.ndim == 3 else onda)


def escaneo(img, escala, sigma_ruido, calidad, rng, sigma_blur=0.0):
    '''sigma_blur: desenfoque Gaussiano HR antes del submuestreo (píxeles HR).
    Atenúa frecuencias por encima del Nyquist LQ que INTER_AREA no puede eliminar.
    sigma_blur=0 → comportamiento original sin cambio.'''
    h, w = img.shape[:2]
    if sigma_blur > 0.0:
        ksize = int(sigma_blur * 6) | 1   # siempre impar
        if img.ndim == 2:
            img = cv2.GaussianBlur(img, (ksize, ksize), sigma_blur)
        else:
            img = cv2.GaussianBlur(img, (ksize, ksize), sigma_blur)
    lq = cv2.resize(img, (w//escala, h//escala), interpolation=cv2.INTER_AREA)
    if sigma_ruido > 0:
        n = rng.normal(0, sigma_ruido, lq.shape[:2]).astype(np.float32)
        lq = lq + (n[..., None] if lq.ndim == 3 else n)
    lq = np.clip(lq, 0, 1)
    if calidad < 100:
        ok, enc = cv2.imencode('.jpg', (lq*255).astype(np.uint8),
                               [int(cv2.IMWRITE_JPEG_QUALITY), int(calidad)])
        if ok:
            lq = cv2.imdecode(enc, cv2.IMREAD_COLOR if lq.ndim == 3 else cv2.IMREAD_GRAYSCALE)
            lq = lq.astype(np.float32)/255.
    return np.clip(lq, 0, 1)

# H_needed se calcula una sola vez al cargar el módulo y se reutiliza en cada
# llamada a H_espectral. Se almacena en coordenadas LQ (ciclos/píxel de 256px).
_PSD = np.load(PSD_OBJETIVO)
_FREQ_LQ  = _PSD['freq'].astype(np.float64)       # shape (N,)
_H_NEEDED = np.clip(10**(_PSD['hueco'] / 2.0), 0.0, 1.0)  # amplitud


def H_espectral(h, w, beta=1.0):
    '''Máscara espectral derivada directamente del objetivo histórico.
    H_beta(f_LQ) = H_needed(f_LQ)^beta
    beta=1.0 → reproduce exactamente el hueco medido.
    beta>1.0 → más agresivo; beta<1.0 → más suave.
    Opera en coordenadas HR y devuelve la máscara rfft2 de tamaño (h, w//2+1).'''
    _, _, rho_HR = malla_rfft(h, w)
    rho_LQ = rho_HR * ESCALA           # mapeo HR→LQ
    H_beta = _H_NEEDED ** beta
    # Interpolar sobre la malla 2D: frecuencias fuera del rango del objetivo
    # reciben H=1.0 (f<min: baja frecuencia, sin atenuar) o H=0.0 (f>0.5 LQ:
    # encima del Nyquist de la imagen LQ, eliminadas en el submuestreo).
    H_mask = np.interp(rho_LQ.ravel(), _FREQ_LQ, H_beta,
                       left=1.0, right=0.0)
    return H_mask.reshape(rho_LQ.shape).astype(np.float32)


def mtf_espectral(img, beta=1.0):
    '''Aplica H_espectral a una imagen HR (2D gris o 3D color).'''
    h, w = img.shape[:2]
    H = H_espectral(h, w, beta)
    if img.ndim == 2:
        return np.fft.irfft2(np.fft.rfft2(img) * H, s=(h, w)).astype(np.float32)
    salida = np.empty_like(img)
    for c in range(img.shape[2]):
        salida[..., c] = np.fft.irfft2(np.fft.rfft2(img[..., c]) * H, s=(h, w))
    return salida


### Comprobaciones unitarias

Cada componente se verifica contra su comportamiento teórico antes de usarlo en la
calibración. El Butterworth debe atenuar exactamente la mitad de la potencia en su
frecuencia de corte, con independencia del orden; el grano debe recuperar la pendiente
`−alpha` en escala log-log; el grano debe ser idéntico en los tres canales; y su amplitud
debe anularse en negros y blancos.

In [ ]:
_WIN = {}
def hann2d(n):
    if n not in _WIN:
        w = np.hanning(n).astype(np.float32); _WIN[n] = np.outer(w, w)
    return _WIN[n]

def rapsd(parche, nbins=None):
    '''Densidad espectral promediada radialmente. Idéntica a la de la Fase 4a.'''
    n = parche.shape[0]
    x = parche - parche.mean(); s = x.std()
    if s < 1e-6: return None, None
    x = (x/s) * hann2d(n)
    P = np.abs(np.fft.fftshift(np.fft.fft2(x)))**2/(n*n)
    c = n//2; yy, xx = np.indices((n, n)); rr = np.hypot(yy - c, xx - c)
    nb = nbins or c
    idx = np.clip((rr/c*nb).astype(np.int32), 0, nb); val = rr <= c
    sm = np.bincount(idx[val], weights=P[val], minlength=nb+1)
    ct = np.bincount(idx[val], minlength=nb+1)
    return (np.arange(1, nb)/nb)*0.5, sm[1:nb]/np.maximum(ct[1:nb], 1)


pruebas = []

# Butterworth: en f = fc la potencia debe caer a la mitad (-0,301 en log10)
ruido = np.random.default_rng(0).normal(size=(512, 512)).astype(np.float32)
for fc, orden in [(0.10, 2), (0.10, 4), (0.20, 3)]:
    f, P = rapsd(mtf_butterworth(ruido, fc, orden))
    caida = float(np.log10(P[np.argmin(abs(f-fc))] / P[np.argmin(abs(f-0.01))]))
    pruebas.append({'prueba': f'Butterworth fc={fc} orden={orden}',
                    'esperado': -0.301, 'medido': caida, 'tol': 0.05})

# Grano: la pendiente en log-log debe recuperar -alpha
for alpha in (1.0, 2.0, 3.0):
    f, P = rapsd(grano_correlacionado(512, 512, alpha, np.random.default_rng(1)))
    m = (f > 0.02) & (f < 0.30)
    pend = float(np.polyfit(np.log10(f[m]), np.log10(P[m]), 1)[0])
    pruebas.append({'prueba': f'Grano alpha={alpha}', 'esperado': -alpha,
                    'medido': pend, 'tol': 0.05})

# Grano acromático: diferencia nula entre canales
plano = np.full((256, 256, 3), 0.5, np.float32)
g = aplicar_grano(plano, 0.05, 2.0, np.random.default_rng(2))
pruebas.append({'prueba': 'Grano acromático (dif. entre canales)', 'esperado': 0.0,
                'medido': float(np.abs(g[..., 0] - g[..., 1]).max()), 'tol': 1e-6})

# Grano nulo en extremos tonales
for L in (0.0, 1.0):
    im = np.full((128, 128, 3), L, np.float32)
    o = aplicar_grano(im, 0.05, 2.0, np.random.default_rng(3))
    pruebas.append({'prueba': f'Grano nulo en L={L:.0f}', 'esperado': 0.0,
                    'medido': float((o-im).std()), 'tol': 1e-6})

PRUEBAS = pd.DataFrame(pruebas)
PRUEBAS['error'] = (PRUEBAS.medido - PRUEBAS.esperado).abs()
PRUEBAS['pasa'] = PRUEBAS.error <= PRUEBAS.tol
display(PRUEBAS.round(5))
assert PRUEBAS.pasa.all(), 'Algún componente no se comporta como debe'
print('Componentes validados.')

## 3. Preparación de la calibración

La calibración compara el espectro que produce el módulo con el objetivo histórico medido
en la Fase 4a. Dos decisiones la hacen viable en tiempo razonable.

**Se calibra en escala de grises.** La RAPSD se mide sobre la luminancia, de modo que
recorrer los tres canales en cada evaluación triplicaría el coste sin cambiar el resultado.
El módulo exportado sí opera en color.

**La transformada de los parches de partida se calcula una sola vez.** El MTF es una
multiplicación en el dominio de Fourier, así que cachear `rfft2` de cada parche ahorra una
transformada por evaluación y por parche.

Se emplean además números aleatorios comunes: todas las evaluaciones usan la misma semilla,
de forma que las diferencias de coste reflejen el cambio de parámetros y no el azar.

**Los límites del grano se fijan por criterio físico, no por conveniencia numérica.** La
RAPSD normaliza cada parche por su desviación típica, de modo que añadir grano intenso de
muy baja frecuencia reduce la potencia normalizada del resto del espectro. El optimizador
puede explotar esa vía para bajar la métrica sin reproducir nada real, y de hecho lo hace si
se le permite: con un exponente de 4,0, la totalidad de la potencia del grano queda en
estructuras mayores de veinte píxeles, que es moteado de gran escala y no emulsión
fotográfica. Restringir el exponente al intervalo entre 0,3 y 1,5 elimina el atajo en su
origen, porque un grano de exponente bajo aporta potencia en alta frecuencia, justo lo
contrario de lo que el optimizador busca. La amplitud se limita en consecuencia a 0,06.

Como el grano deja de servir como palanca, la frecuencia de corte se libera hasta 0,008 para
que sea la MTF quien asuma la atenuación. Conviene recordar que esa frecuencia se expresa en
coordenadas de la imagen de alta resolución: tras el submuestreo se multiplica por el factor
de escala, de modo que un corte de 0,03 equivale a 0,12 ciclos por píxel en la imagen de
salida.

In [ ]:
rng_sel = np.random.default_rng(SEMILLA)
ficheros = listar(DIV2K)
assert ficheros, f'Sin imágenes en {DIV2K}'
sel = [ficheros[i] for i in rng_sel.choice(len(ficheros), min(N_CALIB, len(ficheros)), replace=False)]

HR_GRIS, HR_FFT = [], []
for f in sel:
    img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
    if img is None or min(img.shape[:2]) < LADO_HR:
        continue
    h, w = img.shape[:2]
    y, x = int(rng_sel.integers(0, h-LADO_HR+1)), int(rng_sel.integers(0, w-LADO_HR+1))
    p = img[y:y+LADO_HR, x:x+LADO_HR].astype(np.float32)/255.
    HR_GRIS.append(p); HR_FFT.append(np.fft.rfft2(p))

print(f'{len(HR_GRIS)} parches HR de {LADO_HR}×{LADO_HR} preparados')
assert len(HR_GRIS) >= 6, 'Muy pocos parches: revisar que DIV2K tenga imágenes suficientemente grandes'

In [ ]:
def degradar_gris(idx, p, rng):
    '''Cadena completa sobre un parche en escala de grises con FFT cacheada.'''
    img = np.fft.irfft2(HR_FFT[idx] * H_espectral(LADO_HR, LADO_HR, p['beta']),
                    s=(LADO_HR, LADO_HR)).astype(np.float32)
    img = np.clip(img, 0, 1)
    img = np.clip(aplicar_grano(img, p['sigma_grano'], p['alpha_grano'], rng), 0, 1)
    return escaneo(img, ESCALA, p['sigma_escaneo'], p['jpeg'], rng, p['sigma_blur'])


def perfil_medio(p, semilla):
    filas = []
    for i in range(len(HR_GRIS)):
        lq = degradar_gris(i, p, np.random.default_rng(semilla*1000 + i))
        _, P = rapsd(lq)
        if P is not None:
            filas.append(np.log10(np.maximum(P, 1e-12)))
    return np.median(np.vstack(filas), 0)


def distancia(p, semilla=1):
    '''Función objetivo: distancia log-RAPSD sobre la banda de calibración.'''
    return d_cal(perfil_medio(p, semilla))

### El grano se fija a mano; no se calibra

Las dos ejecuciones de prueba de este notebook dejan un patrón que conviene no ignorar. Con
el grano acotado hasta 0,06 de amplitud, la calibración lo dejó en 0,009 pegado al exponente
máximo. Al ampliar el límite hasta 0,10 para darle más margen, la calibración no lo usó: lo
llevó a **cero**. En ambos casos el optimizador se mueve en la misma dirección, hacia menos
grano, y la razón es estructural y no un problema de rango: cualquier ruido añadido crea un
suelo en alta frecuencia que estorba para igualar una caída tan pronunciada como la del
objetivo histórico. Es la misma lección de la Fase 4a —el ruido del pipeline anterior
enmascaraba la atenuación real— aplicada ahora a la propia calibración. Ampliar de nuevo el
rango no cambiaría la conclusión, porque el óptimo bajo esta métrica es sencillamente no usar
grano.

Ese óptimo numérico entra en conflicto con el propósito físico del módulo: las fotografías
históricas sí tienen grano visible, y un conjunto de entrenamiento sin él no enseña al modelo
a tratarlo. La solución es dejar que la calibración automática resuelva lo que se le da bien
—la envolvente de atenuación, es decir `fc`, `orden`, `sigma_escaneo` y `jpeg`— y fijar el
grano a mano, por inspección visual contra las fotografías históricas, en la sección 5. Los
valores de partida son un punto de arranque razonable, no un resultado calibrado, y se pueden
ajustar y volver a ejecutar desde aquí en función de lo que muestren las figuras.

In [ ]:
# Valores de partida para el grano, fijados por juicio visual y no por optimización.
# Ajustar aquí si la figura de la sección 5 no se parece a las fotografías históricas,
# y volver a ejecutar el notebook desde esta celda.
GRANO_SIGMA_FIJO = 0.052   # amplitud — aumentado de 0.035 para degradación más agresiva
GRANO_ALPHA_FIJO = 1.00    # exponente: bajo = grano fino, alto = moteado de gran escala
GRANO_FIJO = {'sigma_grano': GRANO_SIGMA_FIJO, 'alpha_grano': GRANO_ALPHA_FIJO}

In [ ]:
# beta escala la profundidad del filtro espectral:
# beta=1.0 → reproduce exactamente el hueco histórico medido
# beta>1.0 → degradación más agresiva que el corpus histórico
# beta:       profundidad del filtro espectral (>1 = más agresivo que histórico)
# sigma_blur: desenfoque Gaussiano HR pre-submuestreo (px HR); atenúa alta frec.
# jpeg:       calidad JPEG del escaneo; límite inferior 10 para explorar
#             compresión agresiva (artefacto real de digitalización histórica)
LIMITES = {'beta':        (0.5,  4.0),
           'sigma_blur':  (0.0,  4.0),
           'sigma_escaneo': (0.0, 0.08), 'jpeg': (10, 100)}
CLAVES = list(LIMITES)

def desnormalizar(v):
    '''Los parámetros se optimizan en [0,1] para que Nelder-Mead no favorezca
    a los de mayor rango numérico (jpeg frente a fc). El grano no se optimiza:
    se añade fijo, según lo explicado más arriba.'''
    p = {k: float(LIMITES[k][0] + np.clip(v[i], 0, 1)*(LIMITES[k][1] - LIMITES[k][0]))
         for i, k in enumerate(CLAVES)}
    return {**p, **GRANO_FIJO}

### Suelo estocástico de la métrica

Antes de optimizar conviene saber qué distancia es alcanzable. Evaluar los mismos
parámetros con semillas distintas da la variación irreducible del estimador: ninguna
calibración puede bajar de ahí, y acercarse a ese valor es el criterio razonable de parada.

In [ ]:
P_SONDA = {'beta': 1.0, 'sigma_blur': 0.0,
           'sigma_escaneo': 0.005, 'jpeg': 92, **GRANO_FIJO}
ref = perfil_medio(P_SONDA, 555)
suelo = [float(np.abs(perfil_medio(P_SONDA, s)[BANDA_CAL] - ref[BANDA_CAL]).mean())
         for s in (11, 22, 33)]
SUELO = float(np.mean(suelo))
print(f'suelo estocástico de la métrica: {SUELO:.4f}   (valores: {[round(x,4) for x in suelo]})')

## 4. Calibración

La búsqueda tiene dos etapas. Un sondeo con secuencia de Sobol explora el espacio completo
y evita que el refinamiento quede atrapado cerca del punto de partida; después, un
Nelder-Mead refina desde los dos mejores candidatos. La comprobación posterior se hace con
una semilla distinta de la usada durante la optimización, para descartar que el resultado
se haya ajustado al ruido concreto de la calibración.

In [ ]:
%%time
def coste(v):
    return distancia(desnormalizar(v), semilla=1)

sondeo = qmc.Sobol(d=len(CLAVES), scramble=True, seed=3).random(N_SONDEO)
evaluado = sorted(((coste(v), v) for v in sondeo), key=lambda t: t[0])
print(f'sondeo Sobol ({N_SONDEO} puntos): mejor {evaluado[0][0]:.4f} · peor {evaluado[-1][0]:.4f}')

# Se refinan los N_CANDIDATOS mejores del sondeo, no solo los 2 primeros. La
# distancia ANTES de refinar no predice bien cuál acaba en el mejor mínimo: un
# punto de partida mediocre puede refinar mejor que uno que parecía prometedor.
# Quedarse con demasiado pocos candidatos deja una cuenca mejor sin explorar.
N_CANDIDATOS = 5
mejor = None
candidatos_refinados = []
for d, v in evaluado[:N_CANDIDATOS]:
    r = minimize(coste, v, method='Nelder-Mead',
                 options=dict(maxfev=N_REFINO, xatol=1e-3, fatol=1e-5))
    candidatos_refinados.append(r.fun)
    if mejor is None or r.fun < mejor.fun:
        mejor = r

print(f'refinados: {[round(c,4) for c in sorted(candidatos_refinados)]}')
P_CAL = desnormalizar(mejor.x)
D_CAL = float(mejor.fun)
D_VAL = distancia(P_CAL, semilla=77)
print(f'refinamiento: {D_CAL:.4f} (calibración) · {D_VAL:.4f} (semilla independiente)')

In [ ]:
perfil_cal = perfil_medio(P_CAL, semilla=77)
D_CAL_INFORME, D_CAL_FORMA = d_informe(perfil_cal), d_forma(perfil_cal)

RESUMEN = pd.DataFrame([
    {'magnitud': 'Pipeline tercera entrega', 'calibracion': D_PARTIDA_CAL,
     'informe': D_PARTIDA_INFORME, 'forma': D_PARTIDA_FORMA},
    {'magnitud': 'Módulo calibrado', 'calibracion': D_VAL,
     'informe': D_CAL_INFORME, 'forma': D_CAL_FORMA},
    {'magnitud': 'Suelo estocástico', 'calibracion': SUELO,
     'informe': np.nan, 'forma': np.nan},
])
display(RESUMEN.round(4))
RESUMEN.to_csv(RESULTADOS/'resumen_calibracion.csv', index=False)

PARAMS = pd.DataFrame([{'parametro': k, 'valor': P_CAL[k],
                        'limite_inf': LIMITES[k][0], 'limite_sup': LIMITES[k][1],
                        'en_el_borde': bool(min(abs(P_CAL[k]-LIMITES[k][0]),
                                                abs(P_CAL[k]-LIMITES[k][1]))
                                            < 0.02*(LIMITES[k][1]-LIMITES[k][0]))}
                       for k in CLAVES])
display(PARAMS.round(4))
PARAMS.to_csv(RESULTADOS/'parametros_calibrados.csv', index=False)
print(f'\nReducción sobre la banda de calibración: {100*(1 - D_VAL/D_PARTIDA_CAL):.1f} %')
print(f'Reducción de la distancia de forma      : {100*(1 - D_CAL_FORMA/D_PARTIDA_FORMA):.1f} %')

print(f'\nGrano (fijado a mano, fuera de la optimización): '
      f'sigma={GRANO_SIGMA_FIJO:.3f}  alpha={GRANO_ALPHA_FIJO:.2f}')
print('Se valida por inspección visual en la sección 5, no por esta tabla.')

### Degeneración de la solución

Conviene comprobar si otros juegos de parámetros alcanzan una distancia comparable. La
RAPSD describe la forma de una curva, y distintas combinaciones de frecuencia de corte,
orden e intensidad de grano pueden producir curvas casi idénticas. Si así ocurre, el módulo
queda calibrado espectralmente pero sus parámetros **no deben interpretarse como una
identificación física** del sistema de captura.

Esta comprobación explora candidatos adicionales del sondeo, más allá de los
`N_CANDIDATOS` ya refinados a fondo en la sección anterior, con un presupuesto menor. Si
alguno de ellos igualara o superara a `D_CAL`, sería señal de que la búsqueda principal
tampoco agotó el espacio y convendría subir `N_CANDIDATOS` antes de dar la calibración por
buena.

In [ ]:
alternativas = []
for d, v in evaluado[:8]:
    r = minimize(coste, v, method='Nelder-Mead', options=dict(maxfev=30, xatol=1e-2, fatol=1e-4))
    p = desnormalizar(r.x)
    if r.fun < D_CAL*1.15:
        alternativas.append({**{k: round(p[k], 4) for k in CLAVES}, 'distancia': round(float(r.fun), 4)})

ALT = pd.DataFrame(alternativas).drop_duplicates()
display(ALT)
if len(ALT) and ALT.distancia.min() < D_CAL:
    print(f'\nAVISO: un candidato adicional ({ALT.distancia.min():.4f}) mejora a D_CAL '
          f'({D_CAL:.4f}). Subir N_CANDIDATOS en la celda de calibración y repetir.')
DEGENERADO = len(ALT) > 1 and ALT[CLAVES].std().max() > 0.05
print('Soluciones distintas con distancia equivalente:', 'SÍ' if DEGENERADO else 'no')

## 5. Validación

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(FREQ, LIMPIO, color='0.7', lw=1.0, ls='--', label='DIV2K')
#ax.plot(FREQ, PARTIDA, color=COLORES['inicial'], lw=1.6, label='Pipeline tercera entrega')
ax.plot(FREQ, perfil_cal, color=COLORES['calibrado'], lw=1.6, label='DIV2K degradado')
ax.plot(FREQ, OBJETIVO, color=COLORES['objetivo'], lw=1.8, label='Vintage Degraded Image')
#ax.axvspan(0.10, 0.45, color='0.85', alpha=0.35, lw=0, zorder=0)
#ax.annotate('banda de calibración', (0.21, ax.get_ylim()[1]), fontsize=6.5, color='0.45', ha='center', va='top')
ax.set_xscale('log')
ax.set_xlabel('Frecuencia normalizada (ciclos/píxel)')
ax.set_ylabel('log$_{10}$ densidad espectral de potencia')
ax.legend(frameon=False, loc='lower left', fontsize=7)
ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig(RESULTADOS/'fig_calibracion.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Residuo por banda, antes y después --------------------------------------
BANDAS = [('baja', 0.02, 0.10), ('media', 0.10, 0.25), ('alta', 0.25, 0.45)]
filas = []
for nombre, f0, f1 in BANDAS:
    m = (FREQ >= f0) & (FREQ < f1)
    filas.append({'banda': nombre,
                  'residuo_antes': float((OBJETIVO[m] - PARTIDA[m]).mean()),
                  'residuo_despues': float((OBJETIVO[m] - perfil_cal[m]).mean()),
                  'mejora_abs': float(np.abs(OBJETIVO[m]-PARTIDA[m]).mean()
                                      - np.abs(OBJETIVO[m]-perfil_cal[m]).mean())})
BANDAS_DF = pd.DataFrame(filas)
display(BANDAS_DF.round(4))
BANDAS_DF.to_csv(RESULTADOS/'residuo_por_banda.csv', index=False)

### Comprobación de regresión cromática

El módulo no debe introducir croma. Se mide sobre una sonda acromática construida por
degradación de parches en escala de grises replicados a tres canales: por construcción su
croma de entrada es exactamente nulo, de modo que cualquier valor de C\* a la salida procede
del propio módulo. Es la misma comprobación que la Fase 0 aplicó al modelo ajustado.

In [ ]:
P_COLOR = {**P_CAL, 'escala': ESCALA, 'aniso_prob': 0.0, 'periodico_prob': 0.0}

def degradar_color(gt, p, rng):
    img = np.clip(mtf_espectral(gt, p['beta']), 0, 1)
    img = np.clip(aplicar_grano(img, p['sigma_grano'], p['alpha_grano'], rng), 0, 1)
    return escaneo(img, p['escala'], p['sigma_escaneo'], p['jpeg'], rng, p['sigma_blur'])

croma = []
for i, g in enumerate(HR_GRIS[:8]):
    gt = np.repeat(g[..., None], 3, axis=2)             # croma de entrada nulo
    lq = degradar_color(gt, P_COLOR, np.random.default_rng(500+i))
    lab = cv2.cvtColor((lq*255).astype(np.uint8), cv2.COLOR_BGR2LAB).astype(np.float32)
    a, b = lab[..., 1]-128, lab[..., 2]-128
    croma.append(float(np.sqrt(a**2 + b**2).mean()))

C_MEDIO = float(np.mean(croma))
print(f'C* medio a la salida sobre entrada de croma nulo: {C_MEDIO:.3f}')
print('Referencia: la Fase 0 midió C* = 12,63 en el modelo con sesgo.')
CROMA_OK = C_MEDIO < 1.0
print('Comprobación cromática:', 'OK' if CROMA_OK else 'REVISAR')

### Inspección visual

El ajuste espectral es condición necesaria pero no suficiente. La RAPSD describe cómo se
reparte la potencia entre frecuencias, pero **ignora la fase**, de modo que dos imágenes con
espectros idénticos pueden tener estructuras muy distintas. Una degradación puede por tanto
acertar la métrica y producir imágenes que no se parezcan a una fotografía antigua, y
entrenar sobre ellas enseñaría al modelo a deshacer un defecto inexistente.

La primera figura enfrenta la referencia, la salida del módulo y una fotografía histórica
real a la misma escala, con una ampliación de detalle. La segunda aísla el grano sobre gris
medio para tres exponentes, lo que permite juzgar directamente si la textura corresponde a
emulsión fotográfica o a moteado de gran escala.

In [ ]:
HISTORICAS = DATA["release:vintage_degraded"]
f_hist = listar(HISTORICAS)
print(f'{len(f_hist)} fotografías históricas disponibles para la comparación')


def degradar_color_full(gt, p, rng):
    '''Cadena completa en color, con los mismos parámetros que exporta el módulo.'''
    img = np.clip(mtf_espectral(gt, p['beta']), 0, 1)
    img = np.clip(aplicar_grano(img, p['sigma_grano'], p['alpha_grano'], rng), 0, 1)
    return escaneo(img, p['escala'], p['sigma_escaneo'], p['jpeg'], rng, p['sigma_blur'])


def a_rgb(x):
    return np.clip(x, 0, 1)[..., ::-1]


def zoom(img, lado=72):
    '''Recorte central ampliado sin interpolación, para juzgar la textura.'''
    h, w = img.shape[:2]
    y, x = (h-lado)//2, (w-lado)//2
    return cv2.resize(img[y:y+lado, x:x+lado], (lado*3, lado*3), interpolation=cv2.INTER_NEAREST)


# --- recorto parches HR en color a partir de los mismos ficheros de calibración
HR_COLOR = []
for f in sel[:3]:
    img = cv2.imread(str(f), cv2.IMREAD_COLOR)
    if img is None or min(img.shape[:2]) < LADO_HR:
        continue
    h, w = img.shape[:2]
    y, x = (h-LADO_HR)//2, (w-LADO_HR)//2
    HR_COLOR.append(img[y:y+LADO_HR, x:x+LADO_HR].astype(np.float32)/255.)
print(f'{len(HR_COLOR)} parches en color preparados')

In [ ]:
n = len(HR_COLOR)
fig, axes = plt.subplots(n, 4, figsize=(6.4, 1.75*n))
axes = np.atleast_2d(axes)
titulos = ['Referencia (DIV2K)', 'Módulo calibrado', 'Detalle del módulo', 'Vintage Degraded Image']

for i, hr in enumerate(HR_COLOR):
    ref = cv2.resize(hr, (ANALISIS, ANALISIS), interpolation=cv2.INTER_AREA)
    lq = degradar_color_full(hr, P_COLOR, np.random.default_rng(900+i))

    if f_hist:
        hist = cv2.imread(str(f_hist[i % len(f_hist)]), cv2.IMREAD_COLOR).astype(np.float32)/255.
        s = min(hist.shape[:2])
        hist = cv2.resize(hist[:s, :s], (ANALISIS, ANALISIS), interpolation=cv2.INTER_AREA)
    else:
        hist = np.zeros_like(ref)

    for j, im in enumerate([ref, lq, zoom(lq), hist]):
        ax = axes[i, j]
        ax.imshow(a_rgb(im)); ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(titulos[j], fontsize=7.5)

fig.tight_layout()
fig.savefig(RESULTADOS/'fig_inspeccion_visual.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- El grano aislado sobre gris medio: la MTF y el escaneo se desactivan para
# --- que lo unico visible sea la textura de la emulsion. El rango de comparacion
# --- es ilustrativo (no procede de una optimizacion): sirve para juzgar si el
# --- valor fijado en GRANO_ALPHA_FIJO da grano fino o moteado de gran escala.
cal = GRANO_ALPHA_FIJO
candidatos = sorted({0.5, cal, 1.8})           # dedup por si cal coincide con un extremo
if len(candidatos) < 3:
    candidatos = sorted(set(candidatos) | {cal + 0.6})

fig, axes = plt.subplots(1, len(candidatos), figsize=(6.4, 2.3))
gris = np.full((LADO_HR, LADO_HR, 3), 0.5, np.float32)
for j, alpha in enumerate(candidatos):
    p = {**P_COLOR, 'sigma_grano': GRANO_SIGMA_FIJO, 'alpha_grano': alpha,
         'fc': 0.49, 'orden': 6.0, 'jpeg': 100, 'sigma_escaneo': 0.0}
    out = degradar_color_full(gris, p, np.random.default_rng(11))
    axes[j].imshow(a_rgb(out), vmin=0.30, vmax=0.70)
    etiqueta = f'alpha = {alpha:.2f}'
    if abs(alpha - cal) < 1e-6:
        etiqueta += ' (fijado)'
    axes[j].set_title(etiqueta, fontsize=7.5)
    axes[j].set_xticks([]); axes[j].set_yticks([])

fig.tight_layout()
fig.savefig(RESULTADOS/'fig_grano_aislado.png', bbox_inches='tight')
plt.show()

print(f'Grano fijado: sigma={GRANO_SIGMA_FIJO:.3f}  alpha={GRANO_ALPHA_FIJO:.2f}')
print('Grano de emulsión: exponente bajo, textura fina.')
print('Moteado de gran escala: exponente alto, manchas de decenas de píxeles.')
print('\nSi el panel "fijado" no se parece a la textura de las históricas reales,')
print('ajustar GRANO_SIGMA_FIJO / GRANO_ALPHA_FIJO en la sección 3 y volver a ejecutar')
print('desde ahí. La métrica de calibración no interviene en esta decisión.')

## 6. Exportación del módulo

El módulo `dataset_espectral.py` versionado en la raíz del repo es la fuente de verdad; esta
sección solo comprueba su presencia y exporta los parámetros calibrados. Los valores
puntuales de la calibración se convierten en **rangos** de muestreo aleatorio: entrenar con
un punto fijo produciría un conjunto sin variabilidad, y la lección de las fases 0 a 3 es
precisamente que la falta de variedad en el conjunto de entrenamiento es lo que lleva al
discriminador a aprender una propiedad canónica.

In [ ]:
def rango(valor, ancho_rel, limites):
    lo, hi = limites
    d = ancho_rel*(hi - lo)
    return [round(float(max(lo, valor - d)), 5), round(float(min(hi, valor + d)), 5)]

# Límites de recorte SOLO para dar variedad de muestreo alrededor del grano fijo;
# no proceden de una optimización, así que se declaran aquí de forma explícita.
LIMITES_GRANO_VIS = {'sigma_grano': (0.0, 0.08), 'alpha_grano': (0.3, 1.8)}

PARAMETROS = {
    'escala': ESCALA,
    'beta_rango':          rango(P_CAL['beta'],          0.20, LIMITES['beta']),
    'sigma_blur_rango':    rango(P_CAL['sigma_blur'],    0.20, LIMITES['sigma_blur']),
    'sigma_grano_rango':   rango(GRANO_SIGMA_FIJO,        0.25, LIMITES_GRANO_VIS['sigma_grano']),
    'alpha_grano_rango':   rango(GRANO_ALPHA_FIJO,        0.15, LIMITES_GRANO_VIS['alpha_grano']),
    'sigma_escaneo_rango': rango(P_CAL['sigma_escaneo'], 0.15, LIMITES['sigma_escaneo']),
    'jpeg_rango':          rango(P_CAL['jpeg'],          0.12, LIMITES['jpeg']),
    'aniso_prob': 0.30, 'aniso_rango': [1.0, 1.4],
    'periodico_prob': 0.15, 'periodico_amplitud': [0.005, 0.02], 'periodico_freq': [0.15, 0.45],
    'procedencia': {'objetivo': str(PSD_OBJETIVO),
                    'banda_calibracion': [0.10, 0.45],
                    'distancia_calibrada': D_VAL, 'distancia_partida': D_PARTIDA_CAL,
                    'distancia_informe': D_CAL_INFORME, 'distancia_forma': D_CAL_FORMA,
                    'suelo_estocastico': SUELO,
                    'grano_fijado_a_mano': True,
                    'parches_calibracion': len(HR_GRIS), 'semilla': SEMILLA},
}
(SALIDA/'parametros_degradacion.json').write_text(json.dumps(PARAMETROS, indent=2))
print(json.dumps({k: v for k, v in PARAMETROS.items() if k != 'procedencia'}, indent=2))

In [ ]:
# El código del módulo NO se genera aquí: dataset_espectral.py vive versionado en la
# raíz del repo (es la clase RealESRGANDatasetEspectral que consume el notebook 05).
# Esta celda solo verifica que está presente; recalibrar regenera el JSON, no el .py.
modulo_repo = ROOT / "dataset_espectral.py"
assert modulo_repo.exists() and modulo_repo.stat().st_size > 2000, \
    "Falta dataset_espectral.py en la raíz del repo (es la fuente de verdad de la clase RealESRGANDatasetEspectral)."
print(f"Módulo versionado en el repo: {modulo_repo}  ({modulo_repo.stat().st_size} bytes)")
print("Recalibrar solo regenera parametros_degradacion.json; el código del módulo no se toca aquí.")

### Fragmento de YAML para la Fase 4c

Un detalle que conviene no pasar por alto: `RealESRGANDataset` entrega la imagen de alta
resolución junto con los núcleos, y es `RealESRGANModel` quien aplica la degradación en GPU.
Este módulo, en cambio, resuelve la degradación por completo dentro del dataset y devuelve
el par ya formado. El YAML debe usar por tanto un modelo de datos emparejados y no
`RealESRGANModel`, que volvería a degradar lo ya degradado.

Conviene recordar además las dos incidencias resueltas en fases anteriores: el
`UNetDiscriminatorSN` propio del repositorio de A-ESRGAN oculta al de BasicSR, por lo que en
el YAML debe referenciarse como `UNetDiscriminatorSN_basicsr`; y BasicSR no descubre una
clase de dataset que nadie ha importado, de modo que el lanzamiento necesita el envoltorio
con `runpy` que importe el módulo antes de arrancar el entrenamiento.

In [ ]:
YAML = f'''
datasets:
  train:
    name: DIV2K_espectral
    type: RealESRGANDatasetEspectral          # registrado en dataset_espectral.py
    dataroot_gt: <ruta a DIV2K>
    ruta_parametros: {MODULO_DIR}/parametros_degradacion.json
    io_backend:
      type: disk
    gt_size: {ANALISIS*ESCALA}
    use_hflip: true
    use_rot: true

network_d:
  type: UNetDiscriminatorSN_basicsr           # evita la colisión con la clase de A-ESRGAN
  num_in_ch: 3
  num_feat: 64
  skip_connection: true
'''.strip()

(RESULTADOS/'fragmento_fase4c.yml').write_text(YAML)
print(YAML)

## 7. Síntesis

In [ ]:
UMBRAL_MEJORA = 0.50        # la distancia debe reducirse al menos a la mitad

def sintesis():
    print('CALIBRACIÓN DEL MÓDULO DE DEGRADACIÓN ESPECTRAL\n')
    print(f'{"":34s}{"antes":>10s}{"después":>10s}')
    print(f'  {"banda de calibración (0,10–0,45)":32s}{D_PARTIDA_CAL:>10.4f}{D_VAL:>10.4f}')
    print(f'  {"banda completa (0,02–0,45)":32s}{D_PARTIDA_INFORME:>10.4f}{D_CAL_INFORME:>10.4f}')
    print(f'  {"forma, sin desplazamiento":32s}{D_PARTIDA_FORMA:>10.4f}{D_CAL_FORMA:>10.4f}')
    print(f'\n  suelo estocástico de la métrica : {SUELO:.4f} '
          f'({D_VAL/SUELO:.1f}× por encima)\n')
    for _, f in BANDAS_DF.iterrows():
        marca = ' (diagnóstico)' if f.banda == 'baja' else ''
        print(f'  banda {f.banda:6s} residuo {f.residuo_antes:+.3f} -> {f.residuo_despues:+.3f}'
              f'   mejora {f.mejora_abs:+.3f}{marca}')
    print(f'\n  croma sobre entrada acromática  : C* = {C_MEDIO:.3f}')
    if DEGENERADO:
        print('  aviso: la solución es degenerada; los parámetros calibran el espectro,')
        print('         no identifican el sistema óptico real.')

    ok = (D_VAL < D_PARTIDA_CAL*UMBRAL_MEJORA) and CROMA_OK
    print('\nVEREDICTO')
    if ok:
        print('  El módulo reproduce el espectro del deterioro real mucho mejor que el')
        print('  pipeline anterior y no introduce croma. Procede la Fase 4c.')
    else:
        print('  La calibración no alcanza el criterio sobre la banda 0,10–0,45.')
        if PARAMS.en_el_borde.any():
            pegados = list(PARAMS.loc[PARAMS.en_el_borde, 'parametro'])
            print(f'  Parámetros pegados a su límite: {pegados}. Si son fc, orden o jpeg,')
            print('  ampliar el rango correspondiente y repetir.')
        else:
            print('  Ningún parámetro está en el borde, de modo que el módulo no alcanza la')
            print('  forma objetivo con los fenómenos que modela. Faltaría alguno por incluir.')
    return ok

CALIBRACION_OK = sintesis()

In [ ]:
def inventario_fase4b():
    chk = [
        ('componentes validados', bool(PRUEBAS.pasa.all())),
        ('objetivo de la Fase 4a cargado', len(FREQ) > 0),
        (f'parches de calibración ({len(HR_GRIS)})', len(HR_GRIS) >= 6),
        ('distancia de calibración reducida a la mitad', D_VAL < D_PARTIDA_CAL*UMBRAL_MEJORA),
        ('forma de la curva mejorada', D_CAL_FORMA < D_PARTIDA_FORMA),
        ('sin parámetros pegados al límite', not PARAMS.en_el_borde.any()),
        ('grano fijado a mano (no optimizado)', 'sigma_grano' not in CLAVES),
        ('sin regresión cromática', CROMA_OK),
        ('parámetros exportados', (RESULTADOS/'parametros_degradacion.json').exists()),
        ('fragmento de YAML generado', (RESULTADOS/'fragmento_fase4c.yml').exists()),
        ('figura de calibración', (RESULTADOS/'fig_calibracion.png').exists()),
        ('figuras de inspección visual',
         all((RESULTADOS/f).exists() for f in ('fig_inspeccion_visual.png', 'fig_grano_aislado.png'))),
    ]
    ancho = max(len(t) for t, _ in chk)
    for t, ok in chk: print(f'  {"OK   " if ok else "FALLA"} {t:{ancho}s}')
    print('\nFase 4b', 'COMPLETA' if all(o for _, o in chk) else 'INCOMPLETA')

inventario_fase4b()
print(f'\nResultados en: {RESULTADOS}')

---

### Cómo leer estos resultados

La magnitud que decide es la distancia log-RAPSD sobre la banda 0,10–0,45, comparada con dos
referencias. Por arriba, la del pipeline de la tercera entrega; por abajo, el suelo
estocástico, que es la variación obtenida evaluando los mismos parámetros con semillas
distintas y que ninguna calibración puede mejorar. Una distancia próxima al suelo indica que
el módulo reproduce el espectro objetivo tan bien como el estimador permite distinguir; una
distancia intermedia indica que la forma funcional del módulo no llega a cubrir la del
deterioro real, lo que apuntaría a que falta algún fenómeno por modelar.

Las otras dos distancias sirven para interpretar la anterior. La de banda completa es
comparable con el 0,911 de la Fase 4a y es la que debe citarse en la memoria al enfrentar
ambas fases. La de forma descuenta el desplazamiento vertical medio: si mejora mucho más que
la de banda completa, el módulo está reproduciendo bien el perfil de la curva y lo que queda
es una diferencia de nivel, atribuible a que la normalización por contraste fija la potencia
total y el módulo no puede redistribuirla hacia la banda baja. Ese caso no es un fallo de
calibración, y conviene consignarlo así en lugar de forzar los parámetros para corregirlo.

Si la tabla de degeneración devuelve varias soluciones con distancia equivalente, la lectura
correcta es que la RAPSD restringe la forma de la curva pero no identifica de manera única
los parámetros físicos. Esto no invalida el módulo para su propósito, que es generar una
distribución de entrada realista, pero obliga a describirlo en la memoria como calibrado
espectralmente y no como una medida del sistema óptico que produjo las fotografías.

Conviene vigilar la columna que marca los parámetros pegados a sus límites. Un óptimo en el
borde del rango admisible sugiere que el verdadero está fuera, y en ese caso procede ampliar
el límite correspondiente y repetir antes de dar la calibración por buena.

Por último, los valores puntuales se exportan como rangos de muestreo y no como constantes.
Entrenar con un punto fijo produciría un conjunto sin variabilidad en la degradación, que es
exactamente la condición bajo la cual las fases 0 a 3 documentaron que el discriminador
aprende una propiedad canónica del conjunto en lugar de la tarea. La anchura de los rangos
es una decisión de diseño, no un resultado de la calibración, y como tal debe declararse.

Ninguna de estas magnitudes sustituye a la inspección visual. Si las imágenes que produce el
módulo no se parecen a las históricas reales, la calibración es correcta y el módulo no
sirve, porque el objetivo no es minimizar una distancia sino generar la distribución de
entrada que el modelo encontrará en inferencia. Ante una discrepancia entre la métrica y lo
que muestran las figuras, manda lo segundo.